In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))  # points to src/
from shared_modeling import load_or_create_master_split_ids, run_model_experiment

In [2]:
predictors = ['V2AH01','V2AH02','V2AH03','V2AH04','V2AH05','V2AH06']
df = pd.read_csv("../../Data/V2A.csv", low_memory=False)
df = df[predictors + ['PublicID']]
df = df.replace('R', np.nan)
df[predictors] = df[predictors].apply(pd.to_numeric, errors='coerce')
df

,V2AH01,V2AH02,V2AH03,V2AH04,V2AH05,V2AH06,PublicID
0,2.0,2.0,2.0,2.0,2.0,2.0,00004O
1,2.0,1.0,2.0,2.0,2.0,2.0,00007I
2,2.0,2.0,2.0,2.0,2.0,2.0,00008G
3,2.0,2.0,2.0,2.0,2.0,2.0,00015J
4,2.0,2.0,2.0,2.0,2.0,2.0,00016H
...,...,...,...,...,...,...,...
8729,2.0,2.0,2.0,2.0,2.0,2.0,17349I
8730,2.0,2.0,2.0,2.0,2.0,2.0,17350A
8731,2.0,2.0,2.0,2.0,2.0,2.0,17351V
8732,2.0,2.0,2.0,2.0,2.0,2.0,17352T


In [3]:
df_outcomes = pd.read_csv('../../Data/Modified/Outcome.csv', usecols=['PublicID', 'MH_outcome'])

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)
df_outcomes

,PublicID,MH_outcome
0,00004O,1
1,00007I,1
2,00008G,0
3,00015J,0
4,00016H,1
...,...,...
7736,17349I,1
7737,17350A,1
7738,17351V,0
7739,17352T,1


In [4]:
df = pd.merge(df, df_outcomes, on='PublicID', how='inner')
df

,V2AH01,V2AH02,V2AH03,V2AH04,V2AH05,V2AH06,PublicID,MH_outcome
0,2.0,2.0,2.0,2.0,2.0,2.0,00004O,1
1,2.0,1.0,2.0,2.0,2.0,2.0,00007I,1
2,2.0,2.0,2.0,2.0,2.0,2.0,00008G,0
3,2.0,2.0,2.0,2.0,2.0,2.0,00015J,0
4,2.0,2.0,2.0,2.0,2.0,2.0,00016H,1
...,...,...,...,...,...,...,...,...
7603,2.0,2.0,2.0,2.0,2.0,2.0,17349I,1
7604,2.0,2.0,2.0,2.0,2.0,2.0,17350A,1
7605,2.0,2.0,2.0,2.0,2.0,2.0,17351V,0
7606,2.0,2.0,2.0,2.0,2.0,2.0,17352T,1


In [5]:
X = df.drop(['MH_outcome', 'PublicID'], axis=1)
y = df['MH_outcome']

train_df = df[df['PublicID'].isin(train_ids)].copy()
test_df = df[df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

y.value_counts()

MH_outcome
0    4517
1    3091
Name: count, dtype: int64

In [6]:
# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    predictors
)

Dropping rows with missing values because impute=False (train: 37, test: 4).
Final dataset sizes for LR (impute=False): train=6047, test=1520
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 0.001, 'classifier__l1_ratio': 0.0}
Best Macro F1 Score: 0.4823
Model Coefficients:
num__V2AH01: -0.019615809465943195
num__V2AH02: -0.15307285851594968
num__V2AH03: -0.033584715283407604
num__V2AH04: -0.019853177105385835
num__V2AH05: -0.030115119407741413
num__V2AH06: -0.0589120946062442
Evaluation Metrics for LR with shared preprocessing and macro F1 grid search:
Accuracy: 0.6020
Precision: 0.5742
Recall: 0.5304
F1-score: 0.4814
ROC AUC: 0.5317


In [7]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    predictors
)

Dropping rows with missing values because impute=False (train: 37, test: 4).
Final dataset sizes for RF (impute=False): train=6047, test=1520
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 20, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 500}
Best Macro F1 Score: 0.4823
Feature Importances:
num__V2AH01: 0.04782747924293523
num__V2AH02: 0.728159410988088
num__V2AH03: 0.056146339260578146
num__V2AH04: 0.009608714183140432
num__V2AH05: 0.032783304438122864
num__V2AH06: 0.12547475188713528
Evaluation Metrics for RF with shared preprocessing and macro F1 grid search:
Accuracy: 0.6013
Precision: 0.5726
Recall: 0.5296
F1-score: 0.4801
ROC AUC: 0.5300


In [8]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    predictors
)

Dropping rows with missing values because impute=False (train: 37, test: 4).
Final dataset sizes for XGB (impute=False): train=6047, test=1520
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.01, 'classifier__max_depth': 7, 'classifier__n_estimators': 100, 'classifier__subsample': 0.8}
Best Macro F1 Score: 0.4820
Feature Importances:
num__V2AH01: 0.01205421518534422
num__V2AH02: 0.8340575098991394
num__V2AH03: 0.06234661117196083
num__V2AH04: 0.0
num__V2AH05: 0.03138924017548561
num__V2AH06: 0.06015239283442497
Evaluation Metrics for XGB with shared preprocessing and macro F1 grid search:
Accuracy: 0.6013
Precision: 0.5726
Recall: 0.5296
F1-score: 0.4801
ROC AUC: 0.5306


In [9]:
# Run the SVM pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    predictors
)

Dropping rows with missing values because impute=False (train: 37, test: 4).
Final dataset sizes for SVM (impute=False): train=6047, test=1520
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 100, 'classifier__estimator__gamma': 'scale', 'classifier__estimator__kernel': 'rbf'}
Best Macro F1 Score: 0.4823
Skipping feature-level SVM output to keep notebook output compact.
Evaluation Metrics for SVM with shared preprocessing and macro F1 grid search:
Accuracy: 0.6013
Precision: 0.5726
Recall: 0.5296
F1-score: 0.4801
ROC AUC: 0.5296
